<a href="https://colab.research.google.com/github/aldikayyis/Early-Warning-System-EWS-Volatilitas-Harga-Pangan-Strategis/blob/main/Model_Prediksi_Harga_Pangan_KSP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import timedelta
from sklearn.metrics import mean_absolute_percentage_error

# ==========================================
# 1. PERSIAPAN DATA (SIMULASI TINGKAT DEWA)
# ==========================================
def buat_data_simulasi_ews():
    print("⏳ Menyiapkan Sistem: Generate data riil tiruan (Sesuai tren SP2KP)...")
    # Menggunakan rentang 2 tahun agar model LSTM punya cukup sejarah untuk belajar
    tanggal = pd.date_range(start='2024-05-20', end='2026-05-20', freq='D')
    hari = np.arange(len(tanggal))

    # Rumus meniru persis kenaikan harga menuju angka ~Rp 13.750 di Mei 2026
    harga_dasar = 12200
    tren = hari * 2.1
    musiman = np.sin(hari / 40) * 150
    np.random.seed(42) # Seed biar bentuk grafiknya selalu bagus
    noise = np.random.normal(0, 30, len(tanggal))

    harga_beras = harga_dasar + tren + musiman + noise

    df = pd.DataFrame({'Tanggal': tanggal, 'Harga': harga_beras})
    df.set_index('Tanggal', inplace=True)
    print(f"   -> Berhasil membuat {len(df)} baris data time-series harian bersih!")
    return df

# Eksekusi generator data
df_pangan = buat_data_simulasi_ews()

# ==========================================
# 2. DATA PREPROCESSING
# ==========================================
print("\n⚙️ Melakukan Data Preprocessing...")
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(df_pangan['Harga'].values.reshape(-1, 1))

# Look back disetel 30 hari (1 bulan) untuk belajar pola historis
look_back = 30

def create_sequences(data, time_step):
    X, y = [], []
    for i in range(len(data) - time_step - 1):
        a = data[i:(i + time_step), 0]
        X.append(a)
        y.append(data[i + time_step, 0])
    return np.array(X), np.array(y)

X, y = create_sequences(data_scaled, look_back)
X = X.reshape(X.shape[0], X.shape[1], 1)

train_size = int(len(X) * 0.8)
X_train, X_test = X[0:train_size], X[train_size:len(X)]
y_train, y_test = y[0:train_size], y[train_size:len(y)]

print(f"Bentuk Data Latih: {X_train.shape}")
print(f"Bentuk Data Uji: {X_test.shape}")

# ==========================================
# 3. MEMBANGUN ARSITEKTUR STACKED LSTM
# ==========================================
print("\n🧠 Membangun Model Stacked LSTM...")
model = Sequential()
model.add(Input(shape=(look_back, 1)))
model.add(LSTM(units=50, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(units=50, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(units=1))
model.compile(optimizer='adam', loss='mean_squared_error')

# ==========================================
# 4. TRAINING MODEL
# ==========================================
print("\n🚀 Memulai proses Training...")
# Gunakan epoch 20 untuk tes cepat, bisa dinaikkan ke 50 atau 100 kalau mau akurasi maksimal
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=20, batch_size=16, verbose=1)

# ==========================================
# 5. PREDIKSI & EVALUASI
# ==========================================
print("\n🎯 Melakukan Prediksi...")
train_predict = model.predict(X_train)
test_predict = model.predict(X_test)

# Kembalikan ke skala harga Rupiah
train_predict = scaler.inverse_transform(train_predict)
y_train_asli = scaler.inverse_transform([y_train])
test_predict = scaler.inverse_transform(test_predict)
y_test_asli = scaler.inverse_transform([y_test])

rmse = np.sqrt(np.mean(((test_predict - y_test_asli[0].reshape(-1,1)) ** 2)))
mape = mean_absolute_percentage_error(y_test_asli[0].reshape(-1,1), test_predict) * 100

print(f"\n✅ Evaluasi Selesai!")
print(f"📉 Root Mean Squared Error (RMSE): Rp {rmse:.2f}")
print(f"🎯 Mean Absolute Percentage Error (MAPE): {mape:.2f}%")

# ==========================================
# 6. FORECASTING (PREDIKSI 7 HARI KE DEPAN)
# ==========================================
print("\n🔮 Memprediksi pergerakan harga 7 hari ke depan (Early Warning System)...")

last_sequence = data_scaled[-look_back:]
curr_sequence = last_sequence.reshape(1, look_back, 1)
future_predictions_scaled = []

for _ in range(7):
    next_day_pred = model.predict(curr_sequence, verbose=0)
    future_predictions_scaled.append(next_day_pred[0, 0])
    curr_sequence = np.append(curr_sequence[:, 1:, :], [next_day_pred], axis=1)

future_predictions = scaler.inverse_transform(np.array(future_predictions_scaled).reshape(-1, 1))

# ==========================================
# 7. VISUALISASI DASHBOARD INTERAKTIF (PLOTLY MULTI-PANEL)
# ==========================================
print("\n📈 Membangun Dashboard Interaktif Multi-Panel (HTML)...")

# Bikin layout grid: 1 baris di atas, 2 kolom di bawah
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"colspan": 2}, None],
           [{}, {}]],
    subplot_titles=(
        '<b>1. Pemantauan & EWS Forecast Harga Beras Medium (SP2KP Simulasi)</b>',
        '<b>2. Model Learning Curve (MSE Loss)</b>',
        '<b>3. Distribusi Error Prediksi (Residuals)</b>'
    ),
    vertical_spacing=0.15,
    row_heights=[0.6, 0.4]
)

# --- [PLOT 1] MAIN EWS FORECAST (ATAS) ---
# Aktual (Biru)
fig.add_trace(go.Scatter(x=df_pangan.index, y=df_pangan['Harga'], mode='lines', name='Harga Aktual', line=dict(color='#1f77b4', width=2)), row=1, col=1)

# Uji / Validasi (Oranye)
test_dates = df_pangan.index[len(train_predict) + (look_back * 2) + 1 : len(train_predict) + (look_back * 2) + 1 + len(test_predict)]
fig.add_trace(go.Scatter(x=test_dates, y=test_predict.flatten(), mode='lines', name='Validasi Model', line=dict(color='#ff7f0e', width=2, dash='dash')), row=1, col=1)

# Forecast 7 Hari (Merah)
last_actual_date = df_pangan.index[-1]
future_dates = [last_actual_date + timedelta(days=i) for i in range(1, 8)]
forecast_dates = [last_actual_date] + future_dates
forecast_values = [df_pangan['Harga'].iloc[-1]] + future_predictions.flatten().tolist()
fig.add_trace(go.Scatter(x=forecast_dates, y=forecast_values, mode='lines+markers', name='Forecast 7 Hari Kedepan', line=dict(color='#d62728', width=3), marker=dict(size=8, symbol='diamond')), row=1, col=1)

# Garis Ambang Batas HET
batas_het = 13900
fig.add_hline(y=batas_het, line_dash="dot", line_color="red", annotation_text=f"Batas HET Nasional (Rp {batas_het})", annotation_position="top left", row=1, col=1)

# --- [PLOT 2] TRAINING LOSS CURVE (KIRI BAWAH) ---
fig.add_trace(go.Scatter(y=history.history['loss'], mode='lines', name='Training Loss', line=dict(color='blue', width=2)), row=2, col=1)
fig.add_trace(go.Scatter(y=history.history['val_loss'], mode='lines', name='Validation Loss', line=dict(color='orange', width=2)), row=2, col=1)

# --- [PLOT 3] RESIDUAL HISTOGRAM (KANAN BAWAH) ---
residuals = test_predict.flatten() - y_test_asli[0]
fig.add_trace(go.Histogram(x=residuals, name='Sebaran Error (Rp)', marker_color='purple', opacity=0.7, nbinsx=20), row=2, col=2)

# --- PERCANTIK LAYOUT KESELURUHAN ---
fig.update_layout(
    title=dict(
        text='<b>Dashboard Komprehensif EWS Volatilitas Pangan KSP</b><br><sup>Evaluasi Arsitektur Stacked LSTM</sup>',
        font=dict(size=22)
    ),
    template='plotly_white',
    height=850, # Dipertinggi biar lega
    hovermode='x unified',
    showlegend=False, # Dimatikan agar tidak nutupin grafik (info legend muncul di hover)
    margin=dict(l=40, r=40, t=100, b=40)
)

# Label Axis
fig.update_xaxes(title_text="Tanggal", row=1, col=1)
fig.update_yaxes(title_text="Harga (Rupiah)", row=1, col=1)
fig.update_xaxes(title_text="Epochs", row=2, col=1)
fig.update_yaxes(title_text="Mean Squared Error", row=2, col=1)
fig.update_xaxes(title_text="Error Prediksi (Rupiah)", row=2, col=2)
fig.update_yaxes(title_text="Frekuensi", row=2, col=2)

# Sorot area prediksi dengan bayangan merah muda
fig.add_vrect(
    x0=last_actual_date, x1=forecast_dates[-1],
    fillcolor="red", opacity=0.1, layer="below", line_width=0,
    annotation_text="Zona Prediksi", annotation_position="top left", row=1, col=1
)

# Simpan jadi HTML
nama_file_html = "dashboard_ews_sp2kp_multipanel.html"
fig.write_html(nama_file_html)
print(f"\n🚀 SUCCESS! Dashboard HTML Multi-Panel telah disimpan sebagai '{nama_file_html}'")

⏳ Menyiapkan Sistem: Generate data riil tiruan (Sesuai tren SP2KP)...
   -> Berhasil membuat 731 baris data time-series harian bersih!

⚙️ Melakukan Data Preprocessing...
Bentuk Data Latih: (560, 30, 1)
Bentuk Data Uji: (140, 30, 1)

🧠 Membangun Model Stacked LSTM...

🚀 Memulai proses Training...
Epoch 1/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - loss: 0.0318 - val_loss: 0.0016
Epoch 2/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 0.0044 - val_loss: 7.7554e-04
Epoch 3/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0036 - val_loss: 0.0011
Epoch 4/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0034 - val_loss: 6.5382e-04
Epoch 5/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0038 - val_loss: 6.0239e-04
Epoch 6/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0029 - val_loss: 0.0034
Epoch 7/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0047 - val_loss: 0.0016
Epoch 8/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0030 - val_loss: 6.0476e-04
Epoch 9/20
